## Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import warnings
warnings.filterwarnings("ignore")

## Load Dataset

In [2]:
zomato_data = pd.read_csv(r"C:\Users\Asus\Documents\NLP_based_resturant_recomendation_system\data\raw\zomato_bangalore.csv")

## Dataset Columns & Info

In [3]:
zomato_data.columns

Index(['url', 'address', 'name', 'online_order', 'book_table', 'rate', 'votes',
       'phone', 'location', 'rest_type', 'dish_liked', 'cuisines',
       'approx_cost(for two people)', 'reviews_list', 'menu_item',
       'listed_in(type)', 'listed_in(city)'],
      dtype='object')

In [4]:
zomato_data.shape

(51717, 17)

In [5]:
zomato_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51717 entries, 0 to 51716
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   url                          51717 non-null  object
 1   address                      51717 non-null  object
 2   name                         51717 non-null  object
 3   online_order                 51717 non-null  object
 4   book_table                   51717 non-null  object
 5   rate                         43942 non-null  object
 6   votes                        51717 non-null  int64 
 7   phone                        50509 non-null  object
 8   location                     51696 non-null  object
 9   rest_type                    51490 non-null  object
 10  dish_liked                   23639 non-null  object
 11  cuisines                     51672 non-null  object
 12  approx_cost(for two people)  51371 non-null  object
 13  reviews_list                 51

In [6]:
zomato_data.isnull().sum()

url                                0
address                            0
name                               0
online_order                       0
book_table                         0
rate                            7775
votes                              0
phone                           1208
location                          21
rest_type                        227
dish_liked                     28078
cuisines                          45
approx_cost(for two people)      346
reviews_list                       0
menu_item                          0
listed_in(type)                    0
listed_in(city)                    0
dtype: int64

Famous cuisine/dish of each area

In [7]:
 zomato_data.head()

,url,address,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),reviews_list,menu_item,listed_in(type),listed_in(city)
0,https://www.zomato.com/bangalore/jalsa-banasha...,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,4.1/5,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,"[('Rated 4.0', 'RATED\n A beautiful place to ...",[],Buffet,Banashankari
1,https://www.zomato.com/bangalore/spice-elephan...,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,4.1/5,787,080 41714161,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800,"[('Rated 4.0', 'RATED\n Had been here for din...",[],Buffet,Banashankari
2,https://www.zomato.com/SanchurroBangalore?cont...,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,Yes,No,3.8/5,918,+91 9663487993,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800,"[('Rated 3.0', ""RATED\n Ambience is not that ...",[],Buffet,Banashankari
3,https://www.zomato.com/bangalore/addhuri-udupi...,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Addhuri Udupi Bhojana,No,No,3.7/5,88,+91 9620009302,Banashankari,Quick Bites,Masala Dosa,"South Indian, North Indian",300,"[('Rated 4.0', ""RATED\n Great food and proper...",[],Buffet,Banashankari
4,https://www.zomato.com/bangalore/grand-village...,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Grand Village,No,No,3.8/5,166,+91 8026612447\r\n+91 9901210005,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600,"[('Rated 4.0', 'RATED\n Very good restaurant ...",[],Buffet,Banashankari


## Drop url, listed_in(city), menu_item from Dataset

In [8]:
# location & listed_in(city) contains same info
zomato_data[["address","location","listed_in(city)"]] 

,address,location,listed_in(city)
0,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Banashankari,Banashankari
1,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Banashankari,Banashankari
2,"1112, Next to KIMS Medical College, 17th Cross...",Banashankari,Banashankari
3,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Banashankari,Banashankari
4,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Basavanagudi,Banashankari
...,...,...,...
51712,"Four Points by Sheraton Bengaluru, 43/3, White...",Whitefield,Whitefield
51713,"Number 10, Garudachar Palya, Mahadevapura, Whi...",Whitefield,Whitefield
51714,Sheraton Grand Bengaluru Whitefield Hotel & Co...,Whitefield,Whitefield
51715,Sheraton Grand Bengaluru Whitefield Hotel & Co...,"ITPL Main Road, Whitefield",Whitefield


In [9]:
loc_data = zomato_data[["address","location","listed_in(city)"]] 
loc_data.to_csv(
    r"C:\Users\Asus\Documents\NLP_based_resturant_recomendation_system\data\processed\location_city_mapping.csv",
    index=False
)

In [10]:
zomato_data["location"].unique()

array(['Banashankari', 'Basavanagudi', 'Mysore Road', 'Jayanagar',
       'Kumaraswamy Layout', 'Rajarajeshwari Nagar', 'Vijay Nagar',
       'Uttarahalli', 'JP Nagar', 'South Bangalore', 'City Market',
       'Nagarbhavi', 'Bannerghatta Road', 'BTM', 'Kanakapura Road',
       'Bommanahalli', nan, 'CV Raman Nagar', 'Electronic City', 'HSR',
       'Marathahalli', 'Sarjapur Road', 'Wilson Garden', 'Shanti Nagar',
       'Koramangala 5th Block', 'Koramangala 8th Block', 'Richmond Road',
       'Koramangala 7th Block', 'Jalahalli', 'Koramangala 4th Block',
       'Bellandur', 'Whitefield', 'East Bangalore', 'Old Airport Road',
       'Indiranagar', 'Koramangala 1st Block', 'Frazer Town', 'RT Nagar',
       'MG Road', 'Brigade Road', 'Lavelle Road', 'Church Street',
       'Ulsoor', 'Residency Road', 'Shivajinagar', 'Infantry Road',
       'St. Marks Road', 'Cunningham Road', 'Race Course Road',
       'Commercial Street', 'Vasanth Nagar', 'HBR Layout', 'Domlur',
       'Ejipura', 'Jeevan 

In [11]:
zomato_data["listed_in(city)"].unique()

array(['Banashankari', 'Bannerghatta Road', 'Basavanagudi', 'Bellandur',
       'Brigade Road', 'Brookefield', 'BTM', 'Church Street',
       'Electronic City', 'Frazer Town', 'HSR', 'Indiranagar',
       'Jayanagar', 'JP Nagar', 'Kalyan Nagar', 'Kammanahalli',
       'Koramangala 4th Block', 'Koramangala 5th Block',
       'Koramangala 6th Block', 'Koramangala 7th Block', 'Lavelle Road',
       'Malleshwaram', 'Marathahalli', 'MG Road', 'New BEL Road',
       'Old Airport Road', 'Rajajinagar', 'Residency Road',
       'Sarjapur Road', 'Whitefield'], dtype=object)

In [12]:
# menu_item is empty.
zomato_data[["url","menu_item"]]

,url,menu_item
0,https://www.zomato.com/bangalore/jalsa-banasha...,[]
1,https://www.zomato.com/bangalore/spice-elephan...,[]
2,https://www.zomato.com/SanchurroBangalore?cont...,[]
3,https://www.zomato.com/bangalore/addhuri-udupi...,[]
4,https://www.zomato.com/bangalore/grand-village...,[]
...,...,...
51712,https://www.zomato.com/bangalore/best-brews-fo...,[]
51713,https://www.zomato.com/bangalore/vinod-bar-and...,[]
51714,https://www.zomato.com/bangalore/plunge-sherat...,[]
51715,https://www.zomato.com/bangalore/chime-sherato...,[]


In [13]:
columns_to_drop = ["url","listed_in(city)","menu_item"]
zomato_data.drop(columns=columns_to_drop, inplace=True)

In [14]:
zomato_data.shape

(51717, 14)

In [15]:
zomato_data.columns

Index(['address', 'name', 'online_order', 'book_table', 'rate', 'votes',
       'phone', 'location', 'rest_type', 'dish_liked', 'cuisines',
       'approx_cost(for two people)', 'reviews_list', 'listed_in(type)'],
      dtype='object')

## Custom data-cleaning function for the rate column 

In [16]:
zomato_data["rate"] = pd.to_numeric(
    zomato_data["rate"].replace(["NEW", "-"], np.nan).astype(str).str.split("/").str[0],
    errors="coerce"
)

zomato_data["rate"]

0        4.1
1        4.1
2        3.8
3        3.7
4        3.8
        ... 
51712    3.6
51713    NaN
51714    NaN
51715    4.3
51716    3.4
Name: rate, Length: 51717, dtype: float64

## Custom data-cleaning function for the approx_cost(for two people) 

In [17]:
zomato_data["approx_cost(for two people)"] = pd.to_numeric(
    zomato_data["approx_cost(for two people)"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip(),
    errors="coerce"
)

zomato_data["approx_cost(for two people)"]

0         800.0
1         800.0
2         800.0
3         300.0
4         600.0
          ...  
51712    1500.0
51713     600.0
51714    2000.0
51715    2500.0
51716    1500.0
Name: approx_cost(for two people), Length: 51717, dtype: float64

## Clean online_order & book_table

In [18]:
zomato_data[["online_order","book_table"]]

,online_order,book_table
0,Yes,Yes
1,Yes,No
2,Yes,No
3,No,No
4,No,No
...,...,...
51712,No,No
51713,No,No
51714,No,No
51715,No,Yes


In [19]:
zomato_data["online_order"] = zomato_data["online_order"].map({
    "Yes": 1,
    "No": 0
})

zomato_data["book_table"] = zomato_data["book_table"].map({
    "Yes": 1,
    "No": 0
})
zomato_data[["online_order","book_table"]]

,online_order,book_table
0,1,1
1,1,0
2,1,0
3,0,0
4,0,0
...,...,...
51712,0,0
51713,0,0
51714,0,0
51715,0,1


## Cleaning function for cuisines and dish_liked that have the same structure

In [20]:
zomato_data[["cuisines","dish_liked"]]

,cuisines,dish_liked
0,"North Indian, Mughlai, Chinese","Pasta, Lunch Buffet, Masala Papad, Paneer Laja..."
1,"Chinese, North Indian, Thai","Momos, Lunch Buffet, Chocolate Nirvana, Thai G..."
2,"Cafe, Mexican, Italian","Churros, Cannelloni, Minestrone Soup, Hot Choc..."
3,"South Indian, North Indian",Masala Dosa
4,"North Indian, Rajasthani","Panipuri, Gol Gappe"
...,...,...
51712,Continental,NaN
51713,Finger Food,NaN
51714,Finger Food,NaN
51715,Finger Food,"Cocktails, Pizza, Buttermilk"


In [21]:
def clean_list_column(value):
    if pd.isna(value):
        return []

    items = str(value).split(",")

    cleaned_items = []

    for item in items:
        item = item.strip()

        if item:
            cleaned_items.append(item)

    return cleaned_items

In [22]:
zomato_data["cuisines"] = zomato_data["cuisines"].apply(clean_list_column)

zomato_data["dish_liked"] = zomato_data["dish_liked"].apply(clean_list_column)

zomato_data[["cuisines","dish_liked"]]

,cuisines,dish_liked
0,"[North Indian, Mughlai, Chinese]","[Pasta, Lunch Buffet, Masala Papad, Paneer Laj..."
1,"[Chinese, North Indian, Thai]","[Momos, Lunch Buffet, Chocolate Nirvana, Thai ..."
2,"[Cafe, Mexican, Italian]","[Churros, Cannelloni, Minestrone Soup, Hot Cho..."
3,"[South Indian, North Indian]",[Masala Dosa]
4,"[North Indian, Rajasthani]","[Panipuri, Gol Gappe]"
...,...,...
51712,[Continental],[]
51713,[Finger Food],[]
51714,[Finger Food],[]
51715,[Finger Food],"[Cocktails, Pizza, Buttermilk]"


## Missing Value Imputation

## fillna for rate 

In [32]:
restaurant_mean = (zomato_data.groupby("name")["rate"].transform("mean")) # Case 1: Same restaurant mean rating

location_median = (zomato_data.groupby("location")["rate"] .transform("median")) # Case 2: Location median rating

global_median = zomato_data["rate"].median() # Case 3: Global median (safety fallback)

In [33]:
zomato_data["rate"] = (
    zomato_data["rate"]
    .fillna(restaurant_mean)
    .fillna(location_median)
    .fillna(global_median)
)

In [34]:
zomato_data["rate"].isnull().sum()

np.int64(0)

## fillna for location

In [37]:
zomato_data["location"] = zomato_data["location"].fillna(
    zomato_data["address"]
        .str.split(",")
        .str[-2]
        .str.strip()
)

In [38]:
zomato_data["location"].isnull().sum()

np.int64(0)

## fillna for approx_cost

In [39]:
restaurant_cost = (
    zomato_data.groupby("name")
    ["approx_cost(for two people)"]
    .transform("mean")
)

location_cost = (
    zomato_data.groupby("location")
    ["approx_cost(for two people)"]
    .transform("median")
)

global_cost = (
    zomato_data["approx_cost(for two people)"]
    .median()
)

In [40]:
zomato_data["approx_cost(for two people)"] = (
    zomato_data["approx_cost(for two people)"]
    .fillna(restaurant_cost)
    .fillna(location_cost)
    .fillna(global_cost)
)

In [41]:
zomato_data["approx_cost(for two people)"].isnull().sum()

np.int64(0)

## Review dataset after cleaning

In [42]:
zomato_data.head()

,address,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),reviews_list,listed_in(type)
0,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,1,1,4.1,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"[Pasta, Lunch Buffet, Masala Papad, Paneer Laj...","[North Indian, Mughlai, Chinese]",800.0,"[('Rated 4.0', 'RATED\n A beautiful place to ...",Buffet
1,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,1,0,4.1,787,080 41714161,Banashankari,Casual Dining,"[Momos, Lunch Buffet, Chocolate Nirvana, Thai ...","[Chinese, North Indian, Thai]",800.0,"[('Rated 4.0', 'RATED\n Had been here for din...",Buffet
2,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,1,0,3.8,918,+91 9663487993,Banashankari,"Cafe, Casual Dining","[Churros, Cannelloni, Minestrone Soup, Hot Cho...","[Cafe, Mexican, Italian]",800.0,"[('Rated 3.0', ""RATED\n Ambience is not that ...",Buffet
3,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Addhuri Udupi Bhojana,0,0,3.7,88,+91 9620009302,Banashankari,Quick Bites,[Masala Dosa],"[South Indian, North Indian]",300.0,"[('Rated 4.0', ""RATED\n Great food and proper...",Buffet
4,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Grand Village,0,0,3.8,166,+91 8026612447\r\n+91 9901210005,Basavanagudi,Casual Dining,"[Panipuri, Gol Gappe]","[North Indian, Rajasthani]",600.0,"[('Rated 4.0', 'RATED\n Very good restaurant ...",Buffet


In [44]:
zomato_data.isnull().sum()

address                           0
name                              0
online_order                      0
book_table                        0
rate                              0
votes                             0
phone                          1208
location                          0
rest_type                       227
dish_liked                        0
cuisines                          0
approx_cost(for two people)       0
reviews_list                      0
listed_in(type)                   0
dtype: int64

## Generating restaurants_cleaned.csv

In [46]:
zomato_data.to_csv(
    r"C:\Users\Asus\Documents\NLP_based_resturant_recomendation_system\data\processed\restaurants_cleaned.csv",
    index=False
)

In [ ]:
zomato_data[
    ["name", "food_score"]
].sort_values(
    "food_score",
    ascending=False
).head(20)